## 1. Khai báo thư viện và thiết lập đường dẫn

Notebook này thực hiện phần phân tích mở rộng sau khi đã sinh và lọc luật kết hợp.

Trọng tâm của notebook không phải là sinh thêm luật mới, mà là phân tích sâu hơn các luật đã có bằng cách kết hợp với các thuộc tính mô tả trong dữ liệu như department, aisle, reordered và thời gian mua hàng.

Mục tiêu là làm cho kết quả luật kết hợp có ý nghĩa ứng dụng thực tế hơn, bao gồm:

- Xác định sản phẩm và ngành hàng trung tâm.
- Phân tích các cặp ngành hàng/quầy hàng có liên hệ mua kèm mạnh.
- Đề xuất nhóm ngành hàng nên đặt gần nhau.
- Tìm các luật phù hợp cho combo hoặc khuyến mãi.
- Phân tích sản phẩm được gợi ý theo hành vi mua lại.
- Bổ sung góc nhìn thời gian mua hàng ở mức vừa đủ.

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

cwd = Path.cwd()

if cwd.name.lower() == "notebooks":
    PROJECT_DIR = cwd.parent
else:
    PROJECT_DIR = cwd

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
RESULTS_DIR = PROJECT_DIR / "results"
REPORTS_DIR = PROJECT_DIR / "reports"
IMAGES_DIR = REPORTS_DIR / "images"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("REPORTS_DIR:", REPORTS_DIR)

PROJECT_DIR: d:\HK2_NAM3\KTDL_KPTT\CUOI_KY
PROCESSED_DIR: d:\HK2_NAM3\KTDL_KPTT\CUOI_KY\data\processed
RESULTS_DIR: d:\HK2_NAM3\KTDL_KPTT\CUOI_KY\results
REPORTS_DIR: d:\HK2_NAM3\KTDL_KPTT\CUOI_KY\reports


## 2. Đọc dữ liệu luật kết hợp và dữ liệu có nhãn

Notebook sử dụng các file kết quả đã tạo ở bước sinh luật kết hợp:

| File | Vai trò |
| --- | --- |
| `association_rules_filtered.csv` | Bảng luật kết hợp đã lọc chất lượng |
| `recommendation_lookup.csv` | Bảng luật dạng một sản phẩm đầu vào -> một sản phẩm gợi ý |
| `instacart_subset_for_rules.csv` | Dữ liệu có nhãn dùng để phân tích mở rộng |
| `product_mapping_sample.csv` | Bảng ánh xạ product_id sang product_name, aisle và department |

Trong đó, `association_rules_filtered.csv` là bảng luật chính. Các file còn lại giúp diễn giải luật theo ngữ cảnh kinh doanh.

In [4]:
rules_filtered = pd.read_csv(RESULTS_DIR / "association_rules_filtered.csv")
recommendation_lookup = pd.read_csv(RESULTS_DIR / "recommendation_lookup.csv")
instacart_subset = pd.read_csv(PROCESSED_DIR / "instacart_subset_for_rules.csv")
product_mapping = pd.read_csv(PROCESSED_DIR / "product_mapping_sample.csv")

print("Trạng thái: Load dữ liệu thành công.")
print("rules_filtered:", rules_filtered.shape)
print("recommendation_lookup:", recommendation_lookup.shape)
print("instacart_subset:", instacart_subset.shape)
print("product_mapping:", product_mapping.shape)

display(rules_filtered.head())

Trạng thái: Load dữ liệu thành công.
rules_filtered: (896, 19)
recommendation_lookup: (450, 16)
instacart_subset: (579185, 16)
product_mapping: (3000, 4)


,antecedent_ids,consequent_ids,antecedent_names,consequent_names,support,support_count,confidence,lift,leverage,conviction,recommendation_score,weighted_recommendation_score,antecedent_len,consequent_len,rule_len,antecedent_departments,consequent_departments,antecedent_aisles,consequent_aisles
0,33548,23296,Peach on the Bottom Nonfat Greek Yogurt,Blueberry on the Bottom Nonfat Greek Yogurt,0.001126,85.0,0.384615,123.075293,0.001116,1.619922,47.336651,210.853885,1,1,2,dairy eggs,dairy eggs,yogurt,yogurt
1,23296,33548,Blueberry on the Bottom Nonfat Greek Yogurt,Peach on the Bottom Nonfat Greek Yogurt,0.001126,85.0,0.360169,123.075293,0.001116,1.558340,44.327966,197.452155,1,1,2,dairy eggs,dairy eggs,yogurt,yogurt
2,46149,196,Zero Calorie Cola,Soda,0.001417,107.0,0.543147,55.131632,0.001391,2.167324,29.944592,140.204509,1,1,2,beverages,beverages,soft drinks,soft drinks
3,28465,24799,Icelandic Style Skyr Blueberry Non-fat Yogurt,Vanilla Skyr Nonfat Yogurt,0.001523,115.0,0.344311,67.713674,0.001500,1.517359,23.314588,110.827999,1,1,2,dairy eggs,dairy eggs,yogurt,yogurt
4,"20119, 35221",21709,"Sparkling Water Berry, Lime Sparkling Water",Sparkling Lemon Water,0.001033,78.0,0.520000,48.361921,0.001011,2.060933,25.148199,109.883744,2,1,3,beverages,beverages,water seltzer sparkling water,water seltzer sparkling water


## 3. Kiểm tra cột dữ liệu và xác định chỉ số xếp hạng chính

Trước khi phân tích, cần kiểm tra các cột quan trọng trong dữ liệu. Notebook ưu tiên sử dụng `weighted_recommendation_score` vì đây là chỉ số cải tiến đã được xây dựng ở bước sinh luật.

Nếu cột này không tồn tại, notebook sẽ sử dụng `recommendation_score` thay thế.

Các thuộc tính nhãn được dùng trong notebook gồm:

| Thuộc tính | Ý nghĩa |
| --- | --- |
| `department_name` | Nhóm ngành hàng của sản phẩm |
| `aisle_name` | Quầy hàng hoặc nhóm sản phẩm chi tiết hơn |
| `reordered` | Sản phẩm có được mua lại hay không |
| `order_dow` | Ngày trong tuần khi đơn hàng được đặt |
| `order_hour_of_day` | Giờ trong ngày khi đơn hàng được đặt |

In [6]:
print("Các cột trong rules_filtered:")
print(rules_filtered.columns.tolist())

print("\nCác cột trong recommendation_lookup:")
print(recommendation_lookup.columns.tolist())

print("\nCác cột trong instacart_subset:")
print(instacart_subset.columns.tolist())

dept_col = "department_name" if "department_name" in instacart_subset.columns else "department"
aisle_col = "aisle_name" if "aisle_name" in instacart_subset.columns else "aisle"

score_col = "weighted_recommendation_score"
if score_col not in rules_filtered.columns:
    score_col = "recommendation_score"

print("\nCột department dùng cho phân tích:", dept_col)
print("Cột aisle dùng cho phân tích:", aisle_col)
print("Chỉ số xếp hạng chính:", score_col)

Các cột trong rules_filtered:
['antecedent_ids', 'consequent_ids', 'antecedent_names', 'consequent_names', 'support', 'support_count', 'confidence', 'lift', 'leverage', 'conviction', 'recommendation_score', 'weighted_recommendation_score', 'antecedent_len', 'consequent_len', 'rule_len', 'antecedent_departments', 'consequent_departments', 'antecedent_aisles', 'consequent_aisles']

Các cột trong recommendation_lookup:
['input_product_id', 'input_product_name', 'recommended_product_id', 'recommended_product_name', 'support', 'support_count', 'confidence', 'lift', 'leverage', 'conviction', 'recommendation_score', 'weighted_recommendation_score', 'antecedent_departments', 'consequent_departments', 'antecedent_aisles', 'consequent_aisles']

Các cột trong instacart_subset:
['order_id', 'user_id', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order', 'days_since_prior_order_filled', 'is_first_order', 'product_id', 'product_name', 'aisle_id', 'aisle_name', 'department_id',

## 4. Phân tích top luật theo weighted recommendation score

Phần này phân tích các luật có điểm xếp hạng cao nhất. Đây là các luật được đánh giá tốt theo hướng cân bằng giữa confidence, lift và support_count.

Ý nghĩa của phân tích này là xác định các luật tiêu biểu nhất sau bước cải tiến phương pháp, thay vì chỉ nhìn vào confidence hoặc lift riêng lẻ.

In [7]:
top_weighted_rules = rules_filtered.sort_values(
    [score_col, "confidence", "lift", "support"],
    ascending=False
).head(30)

display_cols = [
    "antecedent_names",
    "consequent_names",
    "support",
    "support_count",
    "confidence",
    "lift",
    "recommendation_score",
    score_col,
    "antecedent_departments",
    "consequent_departments",
    "antecedent_aisles",
    "consequent_aisles"
]

available_display_cols = [col for col in display_cols if col in top_weighted_rules.columns]

display(top_weighted_rules[available_display_cols].head(15))

top_weighted_rules.to_csv(
    RESULTS_DIR / "deep_top_rules_by_weighted_score.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Đã xuất file deep_top_rules_by_weighted_score.csv")

,antecedent_names,consequent_names,support,support_count,confidence,lift,recommendation_score,weighted_recommendation_score,antecedent_departments,consequent_departments,antecedent_aisles,consequent_aisles
0,Peach on the Bottom Nonfat Greek Yogurt,Blueberry on the Bottom Nonfat Greek Yogurt,0.001126,85.0,0.384615,123.075293,47.336651,210.853885,dairy eggs,dairy eggs,yogurt,yogurt
1,Blueberry on the Bottom Nonfat Greek Yogurt,Peach on the Bottom Nonfat Greek Yogurt,0.001126,85.0,0.360169,123.075293,44.327966,197.452155,dairy eggs,dairy eggs,yogurt,yogurt
2,Zero Calorie Cola,Soda,0.001417,107.0,0.543147,55.131632,29.944592,140.204509,beverages,beverages,soft drinks,soft drinks
3,Icelandic Style Skyr Blueberry Non-fat Yogurt,Vanilla Skyr Nonfat Yogurt,0.001523,115.0,0.344311,67.713674,23.314588,110.827999,dairy eggs,dairy eggs,yogurt,yogurt
4,"Sparkling Water Berry, Lime Sparkling Water",Sparkling Lemon Water,0.001033,78.0,0.520000,48.361921,25.148199,109.883744,beverages,beverages,water seltzer sparkling water,water seltzer sparkling water
5,"Sparkling Lemon Water, Lime Sparkling Water",Sparkling Water Grapefruit,0.002172,164.0,0.735426,28.120829,20.680789,105.594980,beverages,beverages,water seltzer sparkling water,water seltzer sparkling water
6,"Sparkling Water Berry, Sparkling Water Grapefruit",Sparkling Lemon Water,0.001218,92.0,0.491979,45.755828,22.510889,102.032843,beverages,beverages,water seltzer sparkling water,water seltzer sparkling water
7,"Sparkling Water Berry, Sparkling Water Grapefruit",Lime Sparkling Water,0.001470,111.0,0.593583,36.296993,21.545274,101.661350,beverages,beverages,water seltzer sparkling water,water seltzer sparkling water
8,"Sparkling Water Berry, Lime Sparkling Water",Sparkling Water Grapefruit,0.001470,111.0,0.740000,28.295727,20.938838,98.799882,beverages,beverages,water seltzer sparkling water,water seltzer sparkling water
9,"Sparkling Water Berry, Sparkling Lemon Water",Lime Sparkling Water,0.001033,78.0,0.604651,36.973807,22.356255,97.684491,beverages,beverages,water seltzer sparkling water,water seltzer sparkling water


Đã xuất file deep_top_rules_by_weighted_score.csv


## 5. Chuẩn hóa hàm xử lý product_id và tạo mapping sản phẩm

Một số cột trong file CSV lưu danh sách product_id dưới dạng chuỗi, ví dụ `"24852"` hoặc `"24852, 21137"`. Vì vậy cần tạo hàm tách chuỗi product_id để phục vụ phân tích sản phẩm ở vế trái và vế phải của luật.

Bước này cũng tạo mapping từ product_id sang tên sản phẩm, department và aisle.

In [8]:
def normalize_product_id(value):
    if pd.isna(value):
        return ""
    value = str(value).strip()
    if value.endswith(".0"):
        value = value[:-2]
    return value

def split_id_string(value):
    if pd.isna(value):
        return []
    parts = str(value).split(",")
    return [normalize_product_id(x) for x in parts if normalize_product_id(x) != ""]

product_mapping["product_id"] = product_mapping["product_id"].apply(normalize_product_id)

product_id_to_name = dict(
    zip(product_mapping["product_id"], product_mapping["product_name"])
)

product_id_to_department = dict(
    zip(product_mapping["product_id"], product_mapping["department_name"])
)

product_id_to_aisle = dict(
    zip(product_mapping["product_id"], product_mapping["aisle_name"])
)

print("Số sản phẩm trong mapping:", len(product_id_to_name))

Số sản phẩm trong mapping: 3000


## 6. Phân tích sản phẩm thường xuất hiện ở vế trái của luật

Sản phẩm ở vế trái của luật có thể được xem là sản phẩm kích hoạt gợi ý. Nếu một sản phẩm xuất hiện nhiều ở vế trái, điều đó cho thấy sản phẩm này thường đóng vai trò bắt đầu cho các quan hệ mua kèm.

Phân tích này giúp xác định các sản phẩm có khả năng kích hoạt mua chéo trong siêu thị.

In [9]:
antecedent_product_rows = []

for _, row in rules_filtered.iterrows():
    for product_id in split_id_string(row["antecedent_ids"]):
        antecedent_product_rows.append({
            "product_id": product_id,
            "product_name": product_id_to_name.get(product_id, product_id),
            "department_name": product_id_to_department.get(product_id, "Unknown"),
            "aisle_name": product_id_to_aisle.get(product_id, "Unknown"),
            "rule_weighted_score": row.get(score_col, row.get("recommendation_score", 0)),
            "confidence": row["confidence"],
            "lift": row["lift"]
        })

antecedent_products_df = pd.DataFrame(antecedent_product_rows)

top_antecedent_products = (
    antecedent_products_df
    .groupby(["product_id", "product_name", "department_name", "aisle_name"])
    .agg(
        num_rules=("product_id", "count"),
        avg_confidence=("confidence", "mean"),
        avg_lift=("lift", "mean"),
        total_weighted_score=("rule_weighted_score", "sum")
    )
    .reset_index()
    .sort_values(["num_rules", "total_weighted_score"], ascending=False)
)

display(top_antecedent_products.head(20))

top_antecedent_products.to_csv(
    RESULTS_DIR / "top_antecedent_products.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Đã xuất file top_antecedent_products.csv")

,product_id,product_name,department_name,aisle_name,num_rules,avg_confidence,avg_lift,total_weighted_score
18,13176,Bag of Organic Bananas,produce,fresh fruits,95,0.262456,3.688643,439.808806
81,21903,Organic Baby Spinach,produce,packaged vegetables fruits,80,0.280781,3.053271,327.669719
264,47209,Organic Hass Avocado,produce,fresh fruits,79,0.312339,3.162297,374.016072
71,21137,Organic Strawberries,produce,fresh fruits,77,0.295494,3.144341,339.180780
109,26209,Limes,produce,fresh fruits,43,0.269333,4.054416,213.793933
268,47766,Organic Avocado,produce,fresh fruits,37,0.313563,3.601365,205.072664
265,47626,Large Lemon,produce,fresh fruits,35,0.285218,3.619405,178.749596
104,24964,Organic Garlic,produce,fresh vegetables,35,0.253166,3.893065,161.712896
249,45007,Organic Zucchini,produce,fresh vegetables,29,0.272850,3.241752,122.493018
102,24852,Banana,produce,fresh fruits,28,0.237890,3.980256,128.788933


Đã xuất file top_antecedent_products.csv


## 7. Phân tích sản phẩm thường được gợi ý ở vế phải

Sản phẩm ở vế phải là sản phẩm được gợi ý mua kèm. Nếu một sản phẩm xuất hiện nhiều ở vế phải, điều đó cho thấy sản phẩm này thường được mua cùng nhiều sản phẩm khác.

Phân tích này giúp xác định các sản phẩm phù hợp để đưa vào khu vực gợi ý, combo hoặc chương trình bán chéo.

In [10]:
consequent_product_rows = []

for _, row in rules_filtered.iterrows():
    for product_id in split_id_string(row["consequent_ids"]):
        consequent_product_rows.append({
            "product_id": product_id,
            "product_name": product_id_to_name.get(product_id, product_id),
            "department_name": product_id_to_department.get(product_id, "Unknown"),
            "aisle_name": product_id_to_aisle.get(product_id, "Unknown"),
            "rule_weighted_score": row.get(score_col, row.get("recommendation_score", 0)),
            "confidence": row["confidence"],
            "lift": row["lift"]
        })

consequent_products_df = pd.DataFrame(consequent_product_rows)

top_recommended_products = (
    consequent_products_df
    .groupby(["product_id", "product_name", "department_name", "aisle_name"])
    .agg(
        num_rules=("product_id", "count"),
        avg_confidence=("confidence", "mean"),
        avg_lift=("lift", "mean"),
        total_weighted_score=("rule_weighted_score", "sum")
    )
    .reset_index()
    .sort_values(["num_rules", "total_weighted_score"], ascending=False)
)

display(top_recommended_products.head(20))

top_recommended_products.to_csv(
    RESULTS_DIR / "top_recommended_products.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Đã xuất file top_recommended_products.csv")

,product_id,product_name,department_name,aisle_name,num_rules,avg_confidence,avg_lift,total_weighted_score
18,24852,Banana,produce,fresh fruits,248,0.285507,1.984285,722.972079
3,13176,Bag of Organic Bananas,produce,fresh fruits,215,0.296131,2.430550,795.631141
10,21137,Organic Strawberries,produce,fresh fruits,103,0.258745,2.973240,391.414808
13,21903,Organic Baby Spinach,produce,packaged vegetables fruits,100,0.245465,3.152057,370.641820
39,47209,Organic Hass Avocado,produce,fresh fruits,79,0.265943,3.810953,390.235830
20,26209,Limes,produce,fresh fruits,27,0.313866,6.383744,272.867086
41,47766,Organic Avocado,produce,fresh fruits,23,0.248651,4.174724,113.885239
37,44632,Sparkling Water Grapefruit,beverages,water seltzer sparkling water,9,0.434703,24.403135,472.416481
40,47626,Large Lemon,produce,fresh fruits,9,0.245771,5.142800,54.079511
14,22935,Organic Yellow Onion,produce,fresh vegetables,8,0.249644,6.855589,64.477084


Đã xuất file top_recommended_products.csv


## 8. Phân tích cặp department trong luật kết hợp

Department là nhãn nhóm ngành hàng. Phân tích luật theo department giúp xác định nhóm hàng nào thường liên kết với nhóm hàng nào.

Đây là bước quan trọng để chuyển kết quả luật kết hợp thành insight kinh doanh, ví dụ đề xuất bán chéo giữa các nhóm hàng hoặc xác định nhóm hàng nên đặt gần nhau.

In [11]:
department_pair_analysis = (
    rules_filtered
    .groupby(["antecedent_departments", "consequent_departments"])
    .agg(
        num_rules=("antecedent_names", "count"),
        avg_support=("support", "mean"),
        avg_confidence=("confidence", "mean"),
        avg_lift=("lift", "mean"),
        avg_weighted_score=(score_col, "mean")
    )
    .reset_index()
    .sort_values(["num_rules", "avg_weighted_score"], ascending=False)
)

display(department_pair_analysis.head(20))

department_pair_analysis.to_csv(
    RESULTS_DIR / "department_pair_analysis.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Đã xuất file department_pair_analysis.csv")

,antecedent_departments,consequent_departments,num_rules,avg_support,avg_confidence,avg_lift,avg_weighted_score
19,produce,produce,601,0.002167,0.281157,3.640717,5.002213
9,dairy eggs,produce,88,0.001913,0.260440,2.406549,3.035268
15,frozen,produce,30,0.001738,0.263389,2.358437,3.026284
3,beverages,beverages,24,0.001870,0.415974,35.414773,69.099950
10,"dairy eggs, produce",produce,23,0.001288,0.311720,3.069588,4.486578
8,dairy eggs,dairy eggs,22,0.001445,0.297008,58.051732,83.768747
20,snacks,produce,19,0.001181,0.257857,2.025720,2.414085
7,canned goods,produce,15,0.001521,0.246745,2.381576,2.819881
11,deli,produce,13,0.001998,0.247354,2.018164,2.485639
4,beverages,produce,13,0.001549,0.251269,1.906332,2.363988


Đã xuất file department_pair_analysis.csv


## 9. Phân tích luật cùng department và khác department

Luật cùng department cho thấy các sản phẩm trong cùng nhóm hàng thường được mua cùng nhau. Luật khác department có ý nghĩa quan trọng hơn trong chiến lược bán chéo, vì nó thể hiện mối quan hệ mua kèm giữa các nhóm hàng khác nhau.

Phân tích này giúp xác định mức độ cross-selling trong tập luật kết hợp.

In [12]:
rules_department_scope = rules_filtered.copy()

rules_department_scope["department_scope"] = np.where(
    rules_department_scope["antecedent_departments"] == rules_department_scope["consequent_departments"],
    "Same department",
    "Cross department"
)

department_scope_summary = (
    rules_department_scope
    .groupby("department_scope")
    .agg(
        num_rules=("antecedent_names", "count"),
        avg_support=("support", "mean"),
        avg_confidence=("confidence", "mean"),
        avg_lift=("lift", "mean"),
        avg_weighted_score=(score_col, "mean")
    )
    .reset_index()
)

display(department_scope_summary)

department_scope_summary.to_csv(
    RESULTS_DIR / "department_scope_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Đã xuất file department_scope_summary.csv")

,department_scope,num_rules,avg_support,avg_confidence,avg_lift,avg_weighted_score
0,Cross department,247,0.00165,0.264013,2.390625,3.045740
1,Same department,649,0.00213,0.286714,6.727461,10.137032


Đã xuất file department_scope_summary.csv


## 10. Phân tích cặp aisle trong luật kết hợp

Aisle là cấp phân loại chi tiết hơn department. Nếu department cho biết nhóm ngành hàng lớn, thì aisle cho biết quầy hàng hoặc nhóm sản phẩm cụ thể hơn.

Phân tích theo aisle giúp hiểu rõ hơn các cặp quầy hàng có liên hệ mua kèm.

In [13]:
aisle_pair_analysis = (
    rules_filtered
    .groupby(["antecedent_aisles", "consequent_aisles"])
    .agg(
        num_rules=("antecedent_names", "count"),
        avg_support=("support", "mean"),
        avg_confidence=("confidence", "mean"),
        avg_lift=("lift", "mean"),
        avg_weighted_score=(score_col, "mean")
    )
    .reset_index()
    .sort_values(["num_rules", "avg_weighted_score"], ascending=False)
)

display(aisle_pair_analysis.head(20))

aisle_pair_analysis.to_csv(
    RESULTS_DIR / "aisle_pair_analysis.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Đã xuất file aisle_pair_analysis.csv")

,antecedent_aisles,consequent_aisles,num_rules,avg_support,avg_confidence,avg_lift,avg_weighted_score
24,fresh fruits,fresh fruits,130,0.002931,0.296946,3.122117,4.700162
29,"fresh fruits, fresh vegetables",fresh fruits,96,0.001361,0.300342,3.469033,4.802893
44,fresh vegetables,fresh fruits,82,0.002721,0.257738,2.364850,3.167144
38,"fresh fruits, packaged vegetables fruits",fresh fruits,70,0.001773,0.301939,3.252492,4.771093
75,packaged vegetables fruits,fresh fruits,48,0.003335,0.253925,2.278097,2.997156
32,"fresh fruits, fresh vegetables",packaged vegetables fruits,37,0.001309,0.263455,3.383073,4.148388
47,"fresh vegetables, packaged vegetables fruits",fresh fruits,32,0.001340,0.300588,2.923838,4.125260
26,fresh fruits,packaged vegetables fruits,24,0.001651,0.238810,3.580876,4.065928
92,water seltzer sparkling water,water seltzer sparkling water,23,0.001890,0.410445,34.557519,66.008448
93,yogurt,fresh fruits,23,0.001397,0.280976,2.251041,3.148385


Đã xuất file aisle_pair_analysis.csv


## 11. Phân tích hành vi mua lại của sản phẩm được gợi ý

Cột `reordered` cho biết sản phẩm có được mua lại hay không. Đây là nhãn hành vi quan trọng trong dữ liệu Instacart.

Phân tích này kiểm tra các sản phẩm được hệ thống gợi ý có tỷ lệ mua lại cao hay không. Nếu một sản phẩm vừa thường được gợi ý, vừa có reordered ratio cao, sản phẩm đó có thể phù hợp cho các chiến dịch gợi ý định kỳ hoặc khuyến mãi lặp lại.

In [14]:
instacart_subset["product_id"] = instacart_subset["product_id"].apply(normalize_product_id)

reordered_analysis = (
    instacart_subset
    .groupby("product_id")
    .agg(
        product_name=("product_name", "first"),
        department_name=(dept_col, "first"),
        aisle_name=(aisle_col, "first"),
        order_count=("order_id", "count"),
        reordered_ratio=("reordered", "mean")
    )
    .reset_index()
)

recommendation_for_reorder = recommendation_lookup.copy()
recommendation_for_reorder["recommended_product_id"] = recommendation_for_reorder["recommended_product_id"].apply(normalize_product_id)

recommended_products_reordered = (
    recommendation_for_reorder
    .merge(
        reordered_analysis,
        left_on="recommended_product_id",
        right_on="product_id",
        how="left"
    )
)

recommended_products_reordered_summary = (
    recommended_products_reordered
    .groupby(["recommended_product_id", "recommended_product_name"])
    .agg(
        num_recommendation_rules=("recommended_product_id", "count"),
        avg_confidence=("confidence", "mean"),
        avg_lift=("lift", "mean"),
        avg_weighted_score=(score_col, "mean"),
        order_count=("order_count", "first"),
        reordered_ratio=("reordered_ratio", "first"),
        department_name=("department_name", "first"),
        aisle_name=("aisle_name", "first")
    )
    .reset_index()
    .sort_values(["reordered_ratio", "avg_weighted_score"], ascending=False)
)

display(recommended_products_reordered_summary.head(20))

recommended_products_reordered_summary.to_csv(
    RESULTS_DIR / "recommended_products_reordered_analysis.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Đã xuất file recommended_products_reordered_analysis.csv")

,recommended_product_id,recommended_product_name,num_recommendation_rules,avg_confidence,avg_lift,avg_weighted_score,order_count,reordered_ratio,department_name,aisle_name
16,24852,Banana,184,0.266516,1.852297,2.560819,10866,0.840880,produce,fresh fruits
19,27845,Organic Whole Milk,2,0.240666,5.179494,5.955005,3509,0.835851,dairy eggs,milk
3,13176,Bag of Organic Bananas,112,0.252523,2.072635,2.760955,9201,0.834475,produce,fresh fruits
33,47209,Organic Hass Avocado,18,0.227922,3.266122,3.680461,5270,0.800190,produce,fresh fruits
18,27086,Half & Half,1,0.247649,10.054945,10.911667,1860,0.787634,dairy eggs,cream
8,21137,Organic Strawberries,39,0.246382,2.831177,3.593799,6572,0.780432,produce,fresh fruits
13,23296,Blueberry on the Bottom Nonfat Greek Yogurt,1,0.384615,123.075293,210.853885,236,0.779661,dairy eggs,yogurt
31,44632,Sparkling Water Grapefruit,4,0.325378,12.441621,22.661108,1975,0.779241,beverages,water seltzer sparkling water
11,21903,Organic Baby Spinach,32,0.225231,2.892236,3.222387,5881,0.775718,produce,packaged vegetables fruits
21,30827,Seedless Cucumbers,1,0.334375,60.701119,95.032921,416,0.771635,produce,packaged produce


Đã xuất file recommended_products_reordered_analysis.csv


## 12. Phân tích ngành hàng trung tâm trong mạng luật kết hợp

Ở phần này, các luật kết hợp được xem như một mạng lưới giữa các department. Mỗi department là một nút, còn mỗi luật từ department A sang department B được xem như một liên kết có hướng.

Một department được xem là trung tâm nếu:

- Xuất hiện nhiều trong luật.
- Có tổng weighted score cao.
- Liên kết với nhiều department khác.
- Đóng vai trò ở cả vế trái và vế phải của luật.

Kết quả phân tích này giúp xác định nhóm hàng cốt lõi trong chiến lược bán chéo và bố trí siêu thị.

In [15]:
def split_label_string(value):
    if pd.isna(value):
        return []
    return [x.strip() for x in str(value).split(",") if x.strip() != ""]

department_edges = []

for _, row in rules_filtered.iterrows():
    antecedent_departments = split_label_string(row["antecedent_departments"])
    consequent_departments = split_label_string(row["consequent_departments"])
    
    for ant_dept in antecedent_departments:
        for con_dept in consequent_departments:
            department_edges.append({
                "antecedent_department": ant_dept,
                "consequent_department": con_dept,
                "support": row["support"],
                "confidence": row["confidence"],
                "lift": row["lift"],
                "weighted_score": row[score_col]
            })

department_edges_df = pd.DataFrame(department_edges)

outgoing_department = (
    department_edges_df
    .groupby("antecedent_department")
    .agg(
        outgoing_rules=("consequent_department", "count"),
        unique_target_departments=("consequent_department", "nunique"),
        avg_outgoing_confidence=("confidence", "mean"),
        avg_outgoing_lift=("lift", "mean"),
        total_outgoing_weighted_score=("weighted_score", "sum")
    )
    .reset_index()
    .rename(columns={"antecedent_department": "department"})
)

incoming_department = (
    department_edges_df
    .groupby("consequent_department")
    .agg(
        incoming_rules=("antecedent_department", "count"),
        unique_source_departments=("antecedent_department", "nunique"),
        avg_incoming_confidence=("confidence", "mean"),
        avg_incoming_lift=("lift", "mean"),
        total_incoming_weighted_score=("weighted_score", "sum")
    )
    .reset_index()
    .rename(columns={"consequent_department": "department"})
)

department_centrality = (
    outgoing_department
    .merge(incoming_department, on="department", how="outer")
    .fillna(0)
)

department_centrality["total_rules"] = (
    department_centrality["outgoing_rules"] +
    department_centrality["incoming_rules"]
)

department_centrality["total_unique_connections"] = (
    department_centrality["unique_target_departments"] +
    department_centrality["unique_source_departments"]
)

department_centrality["total_weighted_score"] = (
    department_centrality["total_outgoing_weighted_score"] +
    department_centrality["total_incoming_weighted_score"]
)

def minmax_scale(series):
    if series.max() == series.min():
        return series * 0
    return (series - series.min()) / (series.max() - series.min())

department_centrality["core_score"] = (
    0.30 * minmax_scale(department_centrality["total_rules"]) +
    0.30 * minmax_scale(department_centrality["total_weighted_score"]) +
    0.20 * minmax_scale(department_centrality["total_unique_connections"]) +
    0.10 * minmax_scale(department_centrality["avg_outgoing_lift"]) +
    0.10 * minmax_scale(department_centrality["avg_incoming_lift"])
)

department_centrality = department_centrality.sort_values(
    "core_score",
    ascending=False
)

display(department_centrality.head(15))

department_centrality.to_csv(
    RESULTS_DIR / "department_core_analysis.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Đã xuất file department_core_analysis.csv")

,department,outgoing_rules,unique_target_departments,avg_outgoing_confidence,avg_outgoing_lift,total_outgoing_weighted_score,incoming_rules,unique_source_departments,avg_incoming_confidence,avg_incoming_lift,total_incoming_weighted_score,total_rules,total_unique_connections,total_weighted_score,core_score
10,produce,634,1,0.282852,3.613981,3156.696772,880.0,12.0,0.277604,3.263288,3898.082979,1514.0,13.0,7054.779751,0.815146
4,dairy eggs,133,2,0.275357,11.725677,2213.207306,23.0,2.0,0.294862,55.964916,1853.824104,156.0,4.0,4067.031411,0.401236
1,beverages,39,2,0.355949,22.609508,1698.929105,24.0,1.0,0.415974,35.414773,1658.398808,63.0,3.0,3357.327913,0.351134
7,frozen,32,2,0.265201,3.803499,162.080643,2.0,1.0,0.292369,25.479428,71.292115,34.0,3.0,233.372758,0.104863
0,bakery,17,2,0.272238,2.904161,63.703067,0.0,0.0,0.000000,0.000000,0.000000,17.0,2.0,63.703067,0.027778
3,canned goods,15,1,0.246745,2.381576,42.298213,0.0,0.0,0.000000,0.000000,0.000000,15.0,1.0,42.298213,0.007308
11,snacks,19,1,0.257857,2.025720,45.867609,0.0,0.0,0.000000,0.000000,0.000000,19.0,1.0,45.867609,0.006555
5,deli,15,1,0.253229,2.132415,39.848047,0.0,0.0,0.000000,0.000000,0.000000,15.0,1.0,39.848047,0.006014
8,meat seafood,14,1,0.250511,2.102843,34.454890,0.0,0.0,0.000000,0.000000,0.000000,14.0,1.0,34.454890,0.005445
6,dry goods pasta,3,1,0.296835,2.155013,8.983095,0.0,0.0,0.000000,0.000000,0.000000,3.0,1.0,8.983095,0.002426


Đã xuất file department_core_analysis.csv


## 13. Đề xuất cặp ngành hàng nên đặt gần nhau

Từ các luật kết hợp giữa department, có thể suy ra các cặp ngành hàng có liên hệ mua kèm mạnh. Nếu hai ngành hàng thường xuyên xuất hiện trong các luật kết hợp và có lift cao, siêu thị có thể cân nhắc đặt chúng gần nhau để tăng khả năng mua kèm.

Phân tích này sử dụng `layout_priority_score`, kết hợp số lượng luật, lift trung bình, confidence trung bình và tổng weighted score.

In [16]:
layout_edges = department_edges_df[
    department_edges_df["antecedent_department"] != department_edges_df["consequent_department"]
].copy()

layout_edges["department_pair"] = layout_edges.apply(
    lambda row: " ↔ ".join(sorted([
        row["antecedent_department"],
        row["consequent_department"]
    ])),
    axis=1
)

layout_pair_analysis = (
    layout_edges
    .groupby("department_pair")
    .agg(
        num_rules=("department_pair", "count"),
        avg_support=("support", "mean"),
        avg_confidence=("confidence", "mean"),
        avg_lift=("lift", "mean"),
        total_weighted_score=("weighted_score", "sum"),
        avg_weighted_score=("weighted_score", "mean")
    )
    .reset_index()
)

layout_pair_analysis["layout_priority_score"] = (
    0.30 * minmax_scale(layout_pair_analysis["num_rules"]) +
    0.25 * minmax_scale(layout_pair_analysis["avg_lift"]) +
    0.25 * minmax_scale(layout_pair_analysis["total_weighted_score"]) +
    0.20 * minmax_scale(layout_pair_analysis["avg_confidence"])
)

layout_pair_analysis = layout_pair_analysis.sort_values(
    "layout_priority_score",
    ascending=False
)

display(layout_pair_analysis.head(20))

layout_pair_analysis.to_csv(
    RESULTS_DIR / "department_layout_recommendation.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Đã xuất file department_layout_recommendation.csv")

,department_pair,num_rules,avg_support,avg_confidence,avg_lift,total_weighted_score,avg_weighted_score,layout_priority_score
5,dairy eggs ↔ produce,111,0.001783,0.271065,2.543936,370.294869,3.335990,0.699261
0,bakery ↔ dairy eggs,1,0.001046,0.247649,10.054945,10.911667,10.911667,0.305619
8,frozen ↔ produce,30,0.001738,0.263389,2.358437,90.788527,3.026284,0.257452
1,bakery ↔ produce,16,0.001476,0.273774,2.457237,52.791400,3.299462,0.227076
7,dry goods pasta ↔ produce,3,0.001271,0.296835,2.155013,8.983095,2.994365,0.221349
11,produce ↔ snacks,19,0.001181,0.257857,2.025720,45.867609,2.414085,0.170084
2,beverages ↔ produce,15,0.001493,0.259910,2.121083,40.530297,2.702020,0.164473
6,deli ↔ produce,15,0.001887,0.253229,2.132415,39.848047,2.656536,0.144383
4,canned goods ↔ produce,15,0.001521,0.246745,2.381576,42.298213,2.819881,0.134122
9,meat seafood ↔ produce,14,0.001590,0.250511,2.102843,34.454890,2.461064,0.128941


Đã xuất file department_layout_recommendation.csv


## 14. Phân tích luật phù hợp để tạo combo hoặc khuyến mãi

Không phải luật nào cũng phù hợp để tạo combo hoặc khuyến mãi. Một luật phù hợp nên có:

- Support_count đủ lớn để không quá hiếm.
- Confidence đủ cao để có khả năng kéo theo sản phẩm vế phải.
- Lift lớn hơn 1 để chứng minh quan hệ mua kèm mạnh hơn ngẫu nhiên.
- Weighted score cao để cân bằng giữa độ phổ biến và mức độ liên hệ.

Phân tích này giúp đề xuất các luật có thể dùng cho combo mua kèm, khuyến mãi chéo hoặc gợi ý sản phẩm.

In [17]:
promotion_candidates = rules_filtered.copy()

promotion_candidates["promotion_type"] = np.where(
    promotion_candidates["antecedent_departments"] == promotion_candidates["consequent_departments"],
    "Same-department bundle",
    "Cross-department bundle"
)

promotion_candidates = promotion_candidates[
    (promotion_candidates["support_count"] >= 100) &
    (promotion_candidates["confidence"] >= 0.20) &
    (promotion_candidates["lift"] >= 1.50)
].copy()

promotion_candidates = promotion_candidates.sort_values(
    [score_col, "confidence", "lift", "support_count"],
    ascending=False
)

promotion_display_cols = [
    "antecedent_names",
    "consequent_names",
    "promotion_type",
    "support",
    "support_count",
    "confidence",
    "lift",
    score_col,
    "antecedent_departments",
    "consequent_departments",
    "antecedent_aisles",
    "consequent_aisles"
]

available_promotion_cols = [
    col for col in promotion_display_cols
    if col in promotion_candidates.columns
]

display(promotion_candidates[available_promotion_cols].head(20))

promotion_candidates.to_csv(
    RESULTS_DIR / "promotion_bundle_candidates.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Đã xuất file promotion_bundle_candidates.csv")

,antecedent_names,consequent_names,promotion_type,support,support_count,confidence,lift,weighted_recommendation_score,antecedent_departments,consequent_departments,antecedent_aisles,consequent_aisles
2,Zero Calorie Cola,Soda,Same-department bundle,0.001417,107.0,0.543147,55.131632,140.204509,beverages,beverages,soft drinks,soft drinks
3,Icelandic Style Skyr Blueberry Non-fat Yogurt,Vanilla Skyr Nonfat Yogurt,Same-department bundle,0.001523,115.0,0.344311,67.713674,110.827999,dairy eggs,dairy eggs,yogurt,yogurt
5,"Sparkling Lemon Water, Lime Sparkling Water",Sparkling Water Grapefruit,Same-department bundle,0.002172,164.0,0.735426,28.120829,105.594980,beverages,beverages,water seltzer sparkling water,water seltzer sparkling water
7,"Sparkling Water Berry, Sparkling Water Grapefruit",Lime Sparkling Water,Same-department bundle,0.001470,111.0,0.593583,36.296993,101.661350,beverages,beverages,water seltzer sparkling water,water seltzer sparkling water
8,"Sparkling Water Berry, Lime Sparkling Water",Sparkling Water Grapefruit,Same-department bundle,0.001470,111.0,0.740000,28.295727,98.799882,beverages,beverages,water seltzer sparkling water,water seltzer sparkling water
10,Total 2% Lowfat Greek Strained Yogurt With Blu...,Total 2% Greek Strained Yogurt with Cherry 5.3 oz,Same-department bundle,0.001523,115.0,0.316804,64.313850,96.853977,dairy eggs,dairy eggs,yogurt,yogurt
12,Vanilla Skyr Nonfat Yogurt,Icelandic Style Skyr Blueberry Non-fat Yogurt,Same-department bundle,0.001523,115.0,0.299479,67.713674,96.397270,dairy eggs,dairy eggs,yogurt,yogurt
13,Rainbow Bell Peppers,Seedless Cucumbers,Same-department bundle,0.001417,107.0,0.334375,60.701119,95.032921,produce,produce,packaged produce,packaged produce
14,Total 2% Greek Strained Yogurt with Cherry 5.3 oz,Total 2% Lowfat Greek Strained Yogurt With Blu...,Same-department bundle,0.001523,115.0,0.309140,64.313850,94.510736,dairy eggs,dairy eggs,yogurt,yogurt
15,"Sparkling Lemon Water, Sparkling Water Grapefruit",Lime Sparkling Water,Same-department bundle,0.002172,164.0,0.546667,33.428113,93.306228,beverages,beverages,water seltzer sparkling water,water seltzer sparkling water


Đã xuất file promotion_bundle_candidates.csv


## 15. Phân loại vai trò ngành hàng

Mỗi ngành hàng có thể đóng vai trò khác nhau trong hệ thống gợi ý và bán chéo:

| Vai trò | Ý nghĩa |
| --- | --- |
| Trigger department | Thường xuất hiện ở vế trái, có khả năng kích hoạt gợi ý |
| Recommended department | Thường xuất hiện ở vế phải, thường được đề xuất mua kèm |
| Core / bridge department | Có vai trò trung tâm, xuất hiện mạnh ở cả hai phía và liên kết nhiều nhóm hàng |
| Balanced department | Có vai trò cân bằng giữa kích hoạt và được gợi ý |

Phân tích này giúp siêu thị hiểu ngành hàng nào nên dùng làm điểm kích hoạt mua hàng, ngành hàng nào nên đưa vào combo và ngành hàng nào là trung tâm trong bố trí cửa hàng.

In [18]:
department_roles = department_centrality.copy()

department_roles["out_in_ratio"] = (
    (department_roles["outgoing_rules"] + 1) /
    (department_roles["incoming_rules"] + 1)
)

def classify_department_role(row):
    if row["core_score"] >= department_roles["core_score"].quantile(0.75):
        return "Core / bridge department"
    elif row["out_in_ratio"] >= 1.5:
        return "Trigger department"
    elif row["out_in_ratio"] <= 0.67:
        return "Recommended department"
    else:
        return "Balanced department"

department_roles["department_role"] = department_roles.apply(
    classify_department_role,
    axis=1
)

department_roles = department_roles.sort_values(
    ["core_score", "total_rules"],
    ascending=False
)

display(
    department_roles[
        [
            "department",
            "department_role",
            "outgoing_rules",
            "incoming_rules",
            "total_unique_connections",
            "total_weighted_score",
            "core_score",
            "out_in_ratio"
        ]
    ].head(20)
)

department_roles.to_csv(
    RESULTS_DIR / "department_role_classification.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Đã xuất file department_role_classification.csv")

,department,department_role,outgoing_rules,incoming_rules,total_unique_connections,total_weighted_score,core_score,out_in_ratio
10,produce,Core / bridge department,634,880.0,13.0,7054.779751,0.815146,0.720772
4,dairy eggs,Core / bridge department,133,23.0,4.0,4067.031411,0.401236,5.583333
1,beverages,Core / bridge department,39,24.0,3.0,3357.327913,0.351134,1.600000
7,frozen,Trigger department,32,2.0,3.0,233.372758,0.104863,11.000000
0,bakery,Trigger department,17,0.0,2.0,63.703067,0.027778,18.000000
3,canned goods,Trigger department,15,0.0,1.0,42.298213,0.007308,16.000000
11,snacks,Trigger department,19,0.0,1.0,45.867609,0.006555,20.000000
5,deli,Trigger department,15,0.0,1.0,39.848047,0.006014,16.000000
8,meat seafood,Trigger department,14,0.0,1.0,34.454890,0.005445,15.000000
6,dry goods pasta,Trigger department,3,0.0,1.0,8.983095,0.002426,4.000000


Đã xuất file department_role_classification.csv


## 16. Phân tích hành vi mua theo ngày trong tuần

Cột `order_dow` cho biết ngày trong tuần khi đơn hàng được đặt. Phân tích này bổ sung góc nhìn hành vi mua hàng theo thời gian ở mức vừa đủ, nhằm hỗ trợ diễn giải kết quả luật kết hợp.

Mục tiêu không phải là chuyển bài thành phân tích thời gian, mà là xác định các nhóm hàng nổi bật theo ngày để hỗ trợ chiến lược gợi ý và khuyến mãi.

In [19]:
if "order_dow" in instacart_subset.columns:
    department_dow_analysis = (
        instacart_subset
        .groupby(["order_dow", dept_col])
        .agg(
            num_order_product_rows=("product_id", "count"),
            num_orders=("order_id", "nunique"),
            reordered_ratio=("reordered", "mean")
        )
        .reset_index()
        .sort_values(["order_dow", "num_order_product_rows"], ascending=[True, False])
    )

    top_department_by_dow = (
        department_dow_analysis
        .groupby("order_dow")
        .head(5)
        .reset_index(drop=True)
    )

    display(top_department_by_dow.head(30))

    top_department_by_dow.to_csv(
        RESULTS_DIR / "top_department_by_dow.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print("Đã xuất file top_department_by_dow.csv")
else:
    print("Không tìm thấy cột order_dow trong dữ liệu.")

,order_dow,department_name,num_order_product_rows,num_orders,reordered_ratio
0,0,produce,46461,10945,0.656787
1,0,dairy eggs,20437,9213,0.685081
2,0,snacks,7070,4431,0.601839
3,0,frozen,7004,4355,0.587807
4,0,beverages,6998,4761,0.674193
5,1,produce,39499,9996,0.661915
6,1,dairy eggs,19227,8726,0.697353
7,1,beverages,7883,4843,0.716352
8,1,snacks,7716,4413,0.648911
9,1,frozen,5614,3597,0.609726


Đã xuất file top_department_by_dow.csv


## 17. Phân tích hành vi mua theo giờ trong ngày

Cột `order_hour_of_day` cho biết giờ đặt hàng trong ngày. Phân tích này giúp quan sát nhóm hàng nào thường xuất hiện nhiều theo từng khung giờ.

Thông tin này có thể hỗ trợ siêu thị thiết kế chương trình gợi ý hoặc khuyến mãi theo thời điểm.

In [20]:
if "order_hour_of_day" in instacart_subset.columns:
    department_hour_analysis = (
        instacart_subset
        .groupby(["order_hour_of_day", dept_col])
        .agg(
            num_order_product_rows=("product_id", "count"),
            num_orders=("order_id", "nunique"),
            reordered_ratio=("reordered", "mean")
        )
        .reset_index()
        .sort_values(["order_hour_of_day", "num_order_product_rows"], ascending=[True, False])
    )

    top_department_by_hour = (
        department_hour_analysis
        .groupby("order_hour_of_day")
        .head(5)
        .reset_index(drop=True)
    )

    display(top_department_by_hour.head(30))

    top_department_by_hour.to_csv(
        RESULTS_DIR / "top_department_by_hour.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print("Đã xuất file top_department_by_hour.csv")
else:
    print("Không tìm thấy cột order_hour_of_day trong dữ liệu.")

,order_hour_of_day,department_name,num_order_product_rows,num_orders,reordered_ratio
0,0,produce,1698,433,0.654888
1,0,dairy eggs,763,355,0.625164
2,0,frozen,282,173,0.581560
3,0,beverages,273,179,0.611722
4,0,snacks,254,162,0.543307
5,1,produce,853,216,0.638921
6,1,dairy eggs,344,166,0.607558
7,1,beverages,162,101,0.660494
8,1,frozen,126,77,0.500000
9,1,snacks,125,79,0.512000


Đã xuất file top_department_by_hour.csv


## 18. Tổng hợp insight ứng dụng thực tế cho siêu thị

Từ các bảng phân tích nâng cao, notebook tổng hợp lại các kết quả có giá trị ứng dụng thực tế:

- Ngành hàng trung tâm trong mạng luật kết hợp.
- Cặp ngành hàng nên cân nhắc đặt gần nhau.
- Luật phù hợp để tạo combo hoặc khuyến mãi.
- Vai trò của từng ngành hàng trong hệ thống gợi ý.

Đây là phần giúp kết quả khai thác luật kết hợp vượt ra khỏi mức kỹ thuật và có thể ứng dụng vào bố trí cửa hàng, bán chéo và thiết kế chương trình khuyến mãi.

In [21]:
top_core_departments = department_centrality.head(5)
top_layout_pairs = layout_pair_analysis.head(5)
top_promotion_candidates = promotion_candidates.head(5)
top_roles = department_roles.head(10)

print("Ngành hàng trung tâm nổi bật:")
display(top_core_departments[["department", "core_score", "total_rules", "total_unique_connections"]])

print("Cặp ngành hàng nên cân nhắc đặt gần nhau:")
display(top_layout_pairs[["department_pair", "num_rules", "avg_confidence", "avg_lift", "layout_priority_score"]])

print("Một số luật phù hợp để tạo combo/khuyến mãi:")
display(
    top_promotion_candidates[
        [
            "antecedent_names",
            "consequent_names",
            "promotion_type",
            "confidence",
            "lift",
            "support_count",
            score_col
        ]
    ]
)

print("Phân loại vai trò ngành hàng:")
display(
    top_roles[
        [
            "department",
            "department_role",
            "outgoing_rules",
            "incoming_rules",
            "core_score"
        ]
    ]
)

Ngành hàng trung tâm nổi bật:


,department,core_score,total_rules,total_unique_connections
10,produce,0.815146,1514.0,13.0
4,dairy eggs,0.401236,156.0,4.0
1,beverages,0.351134,63.0,3.0
7,frozen,0.104863,34.0,3.0
0,bakery,0.027778,17.0,2.0


Cặp ngành hàng nên cân nhắc đặt gần nhau:


,department_pair,num_rules,avg_confidence,avg_lift,layout_priority_score
5,dairy eggs ↔ produce,111,0.271065,2.543936,0.699261
0,bakery ↔ dairy eggs,1,0.247649,10.054945,0.305619
8,frozen ↔ produce,30,0.263389,2.358437,0.257452
1,bakery ↔ produce,16,0.273774,2.457237,0.227076
7,dry goods pasta ↔ produce,3,0.296835,2.155013,0.221349


Một số luật phù hợp để tạo combo/khuyến mãi:


,antecedent_names,consequent_names,promotion_type,confidence,lift,support_count,weighted_recommendation_score
2,Zero Calorie Cola,Soda,Same-department bundle,0.543147,55.131632,107.0,140.204509
3,Icelandic Style Skyr Blueberry Non-fat Yogurt,Vanilla Skyr Nonfat Yogurt,Same-department bundle,0.344311,67.713674,115.0,110.827999
5,"Sparkling Lemon Water, Lime Sparkling Water",Sparkling Water Grapefruit,Same-department bundle,0.735426,28.120829,164.0,105.594980
7,"Sparkling Water Berry, Sparkling Water Grapefruit",Lime Sparkling Water,Same-department bundle,0.593583,36.296993,111.0,101.661350
8,"Sparkling Water Berry, Lime Sparkling Water",Sparkling Water Grapefruit,Same-department bundle,0.740000,28.295727,111.0,98.799882


Phân loại vai trò ngành hàng:


,department,department_role,outgoing_rules,incoming_rules,core_score
10,produce,Core / bridge department,634,880.0,0.815146
4,dairy eggs,Core / bridge department,133,23.0,0.401236
1,beverages,Core / bridge department,39,24.0,0.351134
7,frozen,Trigger department,32,2.0,0.104863
0,bakery,Trigger department,17,0.0,0.027778
3,canned goods,Trigger department,15,0.0,0.007308
11,snacks,Trigger department,19,0.0,0.006555
5,deli,Trigger department,15,0.0,0.006014
8,meat seafood,Trigger department,14,0.0,0.005445
6,dry goods pasta,Trigger department,3,0.0,0.002426


## 19. Tạo bảng tổng hợp kết quả phân tích mở rộng

Bảng tổng hợp giúp kiểm soát toàn bộ phần phân tích mở rộng, bao gồm số luật, số sản phẩm, số cặp department, số cặp aisle, số ngành hàng trung tâm và số đề xuất combo.

Bảng này có thể dùng trong báo cáo để trình bày quy mô kết quả phân tích.

In [22]:
deep_analysis_summary = pd.DataFrame([
    {
        "metric": "Số luật sau lọc",
        "value": len(rules_filtered)
    },
    {
        "metric": "Số luật dùng cho web demo",
        "value": len(recommendation_lookup)
    },
    {
        "metric": "Số sản phẩm xuất hiện ở vế trái",
        "value": top_antecedent_products["product_id"].nunique()
    },
    {
        "metric": "Số sản phẩm được gợi ý ở vế phải",
        "value": top_recommended_products["product_id"].nunique()
    },
    {
        "metric": "Số cặp department trong luật",
        "value": len(department_pair_analysis)
    },
    {
        "metric": "Số cặp aisle trong luật",
        "value": len(aisle_pair_analysis)
    },
    {
        "metric": "Số cặp department đề xuất bố trí",
        "value": len(layout_pair_analysis)
    },
    {
        "metric": "Số luật đề xuất combo/khuyến mãi",
        "value": len(promotion_candidates)
    },
    {
        "metric": "Số department được phân loại vai trò",
        "value": len(department_roles)
    }
])

display(deep_analysis_summary)

deep_analysis_summary.to_csv(
    RESULTS_DIR / "deep_analysis_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Đã xuất file deep_analysis_summary.csv")

,metric,value
0,Số luật sau lọc,896
1,Số luật dùng cho web demo,450
2,Số sản phẩm xuất hiện ở vế trái,311
3,Số sản phẩm được gợi ý ở vế phải,45
4,Số cặp department trong luật,21
5,Số cặp aisle trong luật,97
6,Số cặp department đề xuất bố trí,12
7,Số luật đề xuất combo/khuyến mãi,474
8,Số department được phân loại vai trò,12


Đã xuất file deep_analysis_summary.csv


## 20. Nhận xét tổng quan cho báo cáo

Phần phân tích mở rộng cho thấy dữ liệu có nhãn giúp luật kết hợp có ý nghĩa thực tế hơn. Thay vì chỉ dừng ở việc xác định sản phẩm nào thường mua cùng nhau, notebook đã phân tích thêm các khía cạnh:

- Sản phẩm nào đóng vai trò kích hoạt gợi ý.
- Sản phẩm nào thường được hệ thống đề xuất.
- Ngành hàng nào là trung tâm trong mạng luật kết hợp.
- Cặp ngành hàng nào nên cân nhắc đặt gần nhau.
- Luật nào phù hợp cho combo hoặc khuyến mãi.
- Sản phẩm được gợi ý có liên quan đến hành vi mua lại hay không.
- Một số xu hướng mua hàng theo ngày và giờ.

Những phân tích này giúp kết quả có thể ứng dụng vào gợi ý sản phẩm, bố trí quầy hàng, bán chéo và thiết kế chương trình khuyến mãi trong siêu thị.

In [23]:
print("Nhận xét tổng quan:")
print("Phần phân tích mở rộng đã sử dụng các nhãn department, aisle, reordered và thời gian mua hàng.")
print("Các luật kết hợp được phân tích theo ngữ cảnh kinh doanh thay vì chỉ nhìn vào chỉ số thống kê.")
print("Kết quả có thể hỗ trợ gợi ý sản phẩm, bố trí nhóm hàng, thiết kế combo và chiến lược bán chéo.")

Nhận xét tổng quan:
Phần phân tích mở rộng đã sử dụng các nhãn department, aisle, reordered và thời gian mua hàng.
Các luật kết hợp được phân tích theo ngữ cảnh kinh doanh thay vì chỉ nhìn vào chỉ số thống kê.
Kết quả có thể hỗ trợ gợi ý sản phẩm, bố trí nhóm hàng, thiết kế combo và chiến lược bán chéo.


## Tổng kết bước phân tích mở rộng

Notebook này đã hoàn thành phần phân tích mở rộng dựa trên luật kết hợp và dữ liệu có nhãn.

Các kết quả chính gồm:

- Phân tích top luật theo weighted_recommendation_score.
- Xác định sản phẩm thường xuất hiện ở vế trái của luật.
- Xác định sản phẩm thường được gợi ý ở vế phải.
- Phân tích luật theo department và aisle.
- Phân tích cùng department và cross department.
- Phân tích reordered của sản phẩm được gợi ý.
- Xác định department trung tâm trong mạng luật kết hợp.
- Đề xuất cặp department nên cân nhắc đặt gần nhau.
- Đề xuất luật phù hợp cho combo hoặc khuyến mãi.
- Phân loại vai trò department trong hệ thống gợi ý.
- Bổ sung phân tích hành vi mua theo ngày và giờ ở mức hỗ trợ.

Bước tiếp theo là xây dựng hoặc nâng cấp web demo để đọc các file kết quả này và trình bày trực quan hơn cho người dùng.

In [24]:
aisle_pair_analysis = (
    rules_filtered
    .groupby(["antecedent_aisles", "consequent_aisles"])
    .agg(
        num_rules=("antecedent_names", "count"),
        avg_support=("support", "mean"),
        avg_confidence=("confidence", "mean"),
        avg_lift=("lift", "mean"),
        avg_weighted_score=(score_col, "mean")
    )
    .reset_index()
    .sort_values(["num_rules", "avg_weighted_score"], ascending=False)
)

display(aisle_pair_analysis.head(20))

aisle_pair_analysis.to_csv(
    RESULTS_DIR / "aisle_pair_analysis.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Đã xuất file aisle_pair_analysis.csv")

,antecedent_aisles,consequent_aisles,num_rules,avg_support,avg_confidence,avg_lift,avg_weighted_score
24,fresh fruits,fresh fruits,130,0.002931,0.296946,3.122117,4.700162
29,"fresh fruits, fresh vegetables",fresh fruits,96,0.001361,0.300342,3.469033,4.802893
44,fresh vegetables,fresh fruits,82,0.002721,0.257738,2.364850,3.167144
38,"fresh fruits, packaged vegetables fruits",fresh fruits,70,0.001773,0.301939,3.252492,4.771093
75,packaged vegetables fruits,fresh fruits,48,0.003335,0.253925,2.278097,2.997156
32,"fresh fruits, fresh vegetables",packaged vegetables fruits,37,0.001309,0.263455,3.383073,4.148388
47,"fresh vegetables, packaged vegetables fruits",fresh fruits,32,0.001340,0.300588,2.923838,4.125260
26,fresh fruits,packaged vegetables fruits,24,0.001651,0.238810,3.580876,4.065928
92,water seltzer sparkling water,water seltzer sparkling water,23,0.001890,0.410445,34.557519,66.008448
93,yogurt,fresh fruits,23,0.001397,0.280976,2.251041,3.148385


Đã xuất file aisle_pair_analysis.csv
